# Preprocessing Pipeline

This notebook prepares two types of input data required by the MECR-RAG system.

---

## Section 1 — Triage Guideline Processor

**Data Source:** AEIS Triage Guideline (DOCX file)
**Purpose:** Convert the triage guideline document into a structured, machine-readable JSON format for LLM retrieval during the RAG pipeline.

**Steps:**
1. Convert DOCX to Markdown
2. Split Markdown into individual guideline sections and save as JSON

---

## Section 2 — Medical Case Notes Preprocessing Pipeline

**Data Source:** AE case record PDF files, organised by triage category (Cat 1–5)
**Purpose:** Convert raw AE case records into structured JSON format. This pipeline handles two types of cases:
- **Database cases** — historical cases indexed into the vector database for past case retrieval
- **New cases** — evaluation cases input into the RAG pipeline for triage assessment

**Steps:**
1. Convert PDF case notes to structured CSV (with anonymisation)
2. Manual review of CSV output
3. Convert CSV to structured JSON
4. Post-process JSON files (sex-specific field handling)

> **Note:** The extraction logic in Section 2.1 is calibrated for the Hong Kong AED PDF record format. If records are exported from a different system or in a different layout, the regex patterns and section boundaries will need to be adapted accordingly.

In [ ]:
import os
import time
import json
import re
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Union
from difflib import SequenceMatcher
import pandas as pd
from docx import Document
from docx.table import _Cell, Table
from docx.text.paragraph import Paragraph
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from unstructured.partition.doc import partition_doc
from unstructured.documents.elements import (
    Title,
    NarrativeText,
    ListItem,
    Text,
    Table as UnstructuredTable,
)
from langchain_text_splitters import MarkdownHeaderTextSplitter

## 1 — Triage Guideline Processor

**Data Source:** AEIS Triage Guideline (DOCX)
**Purpose:** Produce a structured JSON file where each entry represents one guideline section, ready for semantic retrieval during the RAG pipeline.

### 1.1 DOCX to Markdown Conversion

Reads the guideline DOCX file and converts it to Markdown, preserving heading structure and table formatting.

In [ ]:
@dataclass
class HeadingInfo:
    """Store heading information including position"""

    level: int
    position: int
    text: str

def get_heading_positions(doc: Document) -> List[HeadingInfo]:
    """Create a list of headings with their positions using python-docx."""
    headings = []

    for pos, para in enumerate(doc.paragraphs):
        text = clean_text(para.text)
        if text and "Heading" in para.style.name:
            try:
                level = int(para.style.name.replace("Heading ", ""))
                heading = HeadingInfo(level=level, position=pos, text=text)
                headings.append(heading)
            except ValueError:
                continue
    return headings

def get_paragraph_level(paragraph: Paragraph) -> int:
    """Determine the nesting level of a paragraph."""
    level = 0
    if hasattr(paragraph._p, "pPr") and paragraph._p.pPr is not None:
        numPr = paragraph._p.pPr.xpath(".//w:numPr")
        if numPr:
            ilvl = numPr[0].xpath(".//w:ilvl")
            if ilvl:
                try:
                    level = int(ilvl[0].get(qn("w:val"))) + 1
                except (ValueError, TypeError):
                    pass
    return level


def clean_text(text: str) -> str:
    """Clean up text by removing extra whitespace and normalizing newlines."""
    text = " ".join(text.split()).strip()
    if text.startswith("{#") and text.endswith(".unnumbered}"):
        text = text.replace("{#", "").replace(".unnumbered}", "")
    return text


def format_cell_content(cell: _Cell) -> str:
    """Format cell content with bullet points in a RAG-friendly format."""
    content_parts = []

    for paragraph in cell.paragraphs:
        text = paragraph.text.strip()
        if not text:
            continue

        level = get_paragraph_level(paragraph)
        if level > 0:
            indent = "  " * (level - 1)
            bullet = "•" if level == 1 else "○"
            content_parts.append(f"{indent}{bullet} {text}")
        else:
            content_parts.append(text)

    return (" ".join(content_parts)).replace("|", "\\|")


def create_markdown_table(table: Table) -> str:
    """Convert a docx table to markdown format optimized for RAG."""
    if not table.rows:
        return ""

    md_lines = []

    # Header row
    header_cells = [format_cell_content(cell) for cell in table.rows[0].cells]
    header = "| " + " | ".join(header_cells) + " |"
    md_lines.append(header)

    # Separator row
    separator = "|" + "|".join("-" * len(cell) for cell in header_cells) + "|"
    md_lines.append(separator)

    # Data rows
    for row in table.rows[1:]:
        cells = [format_cell_content(cell) for cell in row.cells]
        while len(cells) < len(header_cells):
            cells.append("")
        row_md = "| " + " | ".join(cells) + " |"
        md_lines.append(row_md)

    return "\n".join(md_lines)


def convert_doc_to_markdown(input_file: str) -> str:
    """
    Convert DOCX to markdown with proper heading levels and formatted tables.
    """
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found: {input_file}")

    # Get document for heading and table processing
    doc = Document(input_file)

    # Get heading information
    headings = get_heading_positions(doc)
    heading_map = {h.text: h.level for h in headings}

    # Process document with unstructured for general content
    elements = partition_doc(filename=input_file)

    markdown_content = []
    current_pos = 0

    for element in elements:
        text = clean_text(element.text)
        if not text:
            continue

        if isinstance(element, UnstructuredTable):
            # When encounter a table, process it using python-docx
            table_index = len(
                [
                    e
                    for e in elements[: elements.index(element)]
                    if isinstance(e, UnstructuredTable)
                ]
            )
            if table_index < len(doc.tables):
                table_md = create_markdown_table(doc.tables[table_index])
                markdown_content.append(f"\n{table_md}\n")
        else:
            # Handle regular content
            if text in heading_map:
                level = heading_map[text]
                markdown_content.append(f"\n{'#' * level} {text}\n")
            elif isinstance(element, ListItem):
                markdown_content.append(f"- {text}\n")
            else:
                markdown_content.append(f"{text}\n")

        current_pos += 1

    return "\n".join(markdown_content)


def process_and_save_document(input_file: str, output_file: str) -> str:
    """
    Process document and save the result.
    """
    try:
        markdown_content = convert_doc_to_markdown(input_file)

        # Save if output file specified
        if output_file:
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(markdown_content)
            print(f"Markdown saved to: {output_file}")

        return markdown_content

    except Exception as e:
        print(f"Error processing document: {str(e)}")
        raise

### 1.2 Section Splitter

Splits the Markdown document into individual sections based on level-2 headers (`##`), then saves each section as a JSON entry with its title and content.

In [ ]:
def split_triage_guidelines(file_path: str) -> List[Dict]:
    """
    Split a markdown file containing triage guidelines into sections based on level 2 headers (##)

    Args:
        file_path (str): Path to the markdown file

    Returns:
        List[Dict]: List of document chunks with metadata
    """
    # Define headers to split on - only want level 2 headers (##)
    headers_to_split_on = [
        ("##", "section_title"),
    ]

    # Initialize markdown splitter
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False,
    )

    try:
        # Read the markdown file
        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()

        # Split the document
        splits = markdown_splitter.split_text(content)

        print(f"Successfully processed triage guidelines")
        print(f"Total sections found: {len(splits)}")

        return splits

    except Exception as e:
        print(f"Error processing file: {str(e)}")
        return []


def save_sections_to_json(splits: List[Dict], output_file: str):
    """
    Save the splits to a JSON file

    Args:
        splits (List[Dict]): List of document splits
        output_file (str): Path to output JSON file
    """
    sections_data = []

    for split in splits:
        section_data = {
            "section_title": split.metadata.get("section_title", "Untitled Section"),
            "content": split.page_content,
        }
        sections_data.append(section_data)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(sections_data, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(sections_data)} sections to {output_file}")


def load_sections_from_json(input_file: str) -> List[Dict[str, Any]]:
    """
    Load sections from a JSON file

    Args:
        input_file (str): Path to input JSON file

    Returns:
        List[Dict[str, Any]]: List of section data
    """
    with open(input_file, "r", encoding="utf-8") as f:
        sections = json.load(f)

    print(f"Loaded {len(sections)} sections from {input_file}")
    return sections

### 1.3 Execution

Set the file paths in the cell below, then run the two steps sequentially.

In [ ]:
# Define file paths
input_docx = "../data/guideline/triage_guideline_sections.docx"
markdown_output = "../data/guideline/triage_guideline_sections.md"
sections_json = "../data/guideline/triage_guideline_sections.json"

In [ ]:
# Step 1: Convert DOCX to Markdown
print("\n=== Step 1: Converting DOCX to Markdown ===")
try:
    markdown_content = process_and_save_document(input_docx, markdown_output)
    print("Conversion complete!")
except Exception as e:
    print(f"Error in Step 1: {str(e)}")

In [ ]:
# Step 2: Split Markdown into sections
print("\n=== Step 2: Splitting Markdown into sections ===")
sections = split_triage_guidelines(markdown_output)
save_sections_to_json(sections, sections_json)

print("\\nProcess complete! Outputs saved to:")
print(f"- {markdown_output}")
print(f"- {sections_json}")

## 2 — Medical Case Notes Preprocessing Pipeline

**Data Source:** AE case record PDF files, organised into subfolders by triage category (`Cat 1` through `Cat 5`)
**Purpose:** Convert raw AE records into structured JSON case files. The pipeline is used for both:
- **Database cases** — historical cases to be indexed into the vector database
- **New cases** — evaluation cases to be run through the RAG pipeline

> The extraction logic is calibrated for the Hong Kong AED clinical documentation form. Records from a different source system will likely require adjustments to the section patterns and regex in Section 2.1.

### 2.1 PDF to CSV Conversion

Extracts structured clinical data from each AE case PDF and saves it to CSV format. One CSV file is produced per input subdirectory.

Set `root_dir` to the root folder containing your `pdf/` subdirectory before running.

In [ ]:
import pdfplumber
import pandas as pd
import re
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
root_dir = "PATH_TO_CASE_NOTES_ROOT"

In [ ]:
def extract_ae_number_from_pdf(pdf_path):
    """
    Extract AE number from PDF using the same pattern as duplicate detection
    Looking for pattern: Case no: AE24200435(X) where X can be ANY content
    """
    # FIXED regex to handle any content in parentheses
    case_pattern = r'Case\s+no:\s*(AE\d+(?:\([^)]+\))?)'
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Check first few pages (case number likely on first page)
            for page_num, page in enumerate(pdf.pages[:3]):
                text = page.extract_text()
                if text:
                    # Search for case number pattern
                    match = re.search(case_pattern, text, re.IGNORECASE)
                    if match:
                        ae_number = match.group(1)
                        return ae_number
            
            return None
            
    except Exception as e:
        print(f"Error extracting AE number from {pdf_path}: {e}")
        return None

print("✅ AE number extraction function defined")

In [ ]:
def anonymize_medical_note(text):
    """
    Remove confidential information from medical notes while preserving content structure.
    Keeps staff names intact.

    Args:
        text (str): Raw text from the medical note
    Returns:
        str: Partially anonymized text with staff names preserved
    """
    # Split text into lines for processing
    lines = text.split("\n")
    cleaned_lines = []

    # Flag to track if we're in the header section
    in_header = True

    # Footer patterns
    footer_patterns = [
        r"This A&E Clinical Documentation Form is intended for.*",
        r"此乃急症室臨床記錄.*",
        r"For professional communication only\.",
        r"只供醫護人員參考。",
        r"Generated on.*Page \d+ of \d+",
        r"-- End --",
    ]

    for line in lines:
        # Skip empty lines
        if not line.strip():
            continue

        # Check if we've reached the end of the header
        if in_header and line.strip().startswith("Sex:"):
            in_header = False

        # Skip if we're still in the header
        if in_header:
            continue

        # Skip if line matches any footer pattern
        if any(re.match(pattern, line.strip()) for pattern in footer_patterns):
            continue

        # No longer replacing staff names - they will be preserved in the output
        cleaned_lines.append(line)

    # Join lines back together
    cleaned_text = "\n".join(cleaned_lines)

    # Remove any multiple consecutive newlines
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)

    return cleaned_text.strip()

def remove_repeating_headers(text):
    """
    Remove repeating header sections that are preceded by a blank line.
    Keep only the first occurrence of the header.

    Args:
        text (str): Text with potentially repeating headers
    Returns:
        str: Text with only the first header occurrence
    """
    # Extract the first occurrence of the header (first two lines)
    lines = text.split("\n")
    if len(lines) < 2:
        return text

    # Get the first two lines as the header pattern
    header_line1 = lines[0]
    header_line2 = lines[1]

    # Create a pattern to match repeating headers preceded by a blank line
    header_pattern = (
        r"(\n\s*\n)" + re.escape(header_line1) + r"\n" + re.escape(header_line2)
    )

    # Replace all occurrences with an empty string (removing both blank line and header)
    cleaned_text = re.sub(header_pattern, "", text)

    return cleaned_text

print("✅ Text processing functions defined")

In [ ]:
def extract_medical_info(text):
    """
    Extract relevant medical information from anonymized text and return as a dictionary
    """
    info = {}

    # Extract key sections first for better organization
    def extract_section(section_name, text):
        section_patterns = {
            "triage": r"Triage\n(.*?)(?:Consultation|Investigation)",
            "vital_signs": r"Vital sign and observations(.*?)(?:Consultation|Investigation)",
            "consultation": r"Consultation\n(.*?)(?:Investigation|Reassessment|Re-Assess|Discharge)",
            "allergy": r"Allergy / Alert Information\n(.*?)(?=\nDiagnosis|\nRecent COVID-19 Information|\nTriage)",
        }

        pattern = section_patterns.get(section_name.lower())
        if not pattern:
            return None

        match = re.search(pattern, text, re.DOTALL)
        return match.group(1) if match else None

    # Extract main sections
    triage_section = extract_section("triage", text)
    vital_section = extract_section("vital_signs", text)
    consultation_section = extract_section("consultation", text)
    allergy_section = extract_section("allergy", text)

    # Basic Demographics
    sex_match = re.search(r"Sex:\s*([MF])", text)
    info["Sex"] = sex_match.group(1) if sex_match else None

    age_match = re.search(r"Age:\s*(\d+)(m)?", text)
    if age_match:
        age_value = age_match.group(1)
        # group(2) could be None if the optional group didn't match
        # Using the or operator to handle this case safely
        age_unit = age_match.group(2) or ""

        # If the unit is "m", it's months, otherwise it's years
        if age_unit == "m":
            info["Age"] = f"{age_value} months"  # Age in months
        else:
            info["Age"] = f"{age_value} years"  # Age in years (no suffix)
    else:
        info["Age"] = None

    # Triage Nurse
    triage_nurse_match = re.search(
        r"Triage time:.*?by\s+([A-Z]+(?:,\s*[A-Z\s]+)+)", triage_section
    )
    info["Triage Nurse"] = (
        triage_nurse_match.group(1).strip() if triage_nurse_match else None
    )

    # Case Doctor
    case_doctor_match = re.search(
        r"Case doctor:\s*([^:\n]+?)(?:\n|$)", consultation_section
    )
    info["Case Doctor"] = (
        case_doctor_match.group(1).strip() if case_doctor_match else None
    )

    # Allergy and Alert
    if allergy_section:
        # Join lines first to handle text split across multiple lines
        normalized_section = re.sub(r"\n\s*", " ", allergy_section)

        # Define the section boundaries based on the known order: Allergy → ADR → Alert
        sections = {
            "Allergy": {"start": "Allergy:", "end": ["ADR:", "Alert:"]},
            "ADR": {"start": "ADR:", "end": ["Alert:"]},
            "Alert": {"start": "Alert:", "end": []},  # No boundary for the last section
        }

        for item_type, boundaries in sections.items():
            start_pattern = re.escape(boundaries["start"])

            # Find the start position of this section
            start_match = re.search(start_pattern, normalized_section)
            if not start_match:
                info[item_type] = None
                continue

            start_pos = start_match.start()

            # Find the end position (start of the next section)
            end_pos = len(normalized_section)
            for end_marker in boundaries["end"]:
                end_match = re.search(re.escape(end_marker), normalized_section)
                if end_match and end_match.start() > start_pos:
                    end_pos = min(end_pos, end_match.start())

            # Extract the section text
            section_text = normalized_section[start_pos:end_pos].strip()

            # Remove the section heading
            section_content = section_text.replace(boundaries["start"], "").strip()

            # Split by semicolon and clean up
            if section_content:
                items = re.split(r";\s*", section_content)
                cleaned_items = [item.strip() for item in items if item.strip()]
                info[item_type] = "; ".join(cleaned_items)
            else:
                info[item_type] = None
    else:
        info["Allergy"] = None
        info["ADR"] = None
        info["Alert"] = None

    # Diagnosis
    diagnosis_match = re.search(
        r"Diagnosis\n\s*(.+?)(?=\n\(\? : Provisional\)|\nTriage|$)",
        text,
        re.DOTALL,
    )
    if diagnosis_match:
        diagnosis_text = diagnosis_match.group(1).strip()
        # Remove "Principal: " prefix if present
        diagnosis_text = re.sub(r"^Principal:\s*", "", diagnosis_text)
        info["Diagnosis"] = diagnosis_text
    else:
        info["Diagnosis"] = None

    # Triage Info
    triage_cat_match = re.search(r"Triage category:\s*(\d)", triage_section)
    info["Triage Category"] = triage_cat_match.group(1) if triage_cat_match else None

    # Ambulatory Status and Fall Risk
    amb_status_match = re.search(
        r"Amb status:\s*(.+?)(?=Risk of fall:|$)", triage_section
    )
    info["Ambulatory status"] = (
        amb_status_match.group(1).strip() if amb_status_match else None
    )

    fall_risk_match = re.search(r"Risk of fall:\s*(.+?)(?=\n|$)", triage_section)
    info["Risk of fall"] = fall_risk_match.group(1).strip() if fall_risk_match else None

    # Communication and Informant
    informant_match = re.search(
        r"Informant:\s*(.+?)(?=Communication|$)", triage_section
    )
    info["Informant"] = informant_match.group(1).strip() if informant_match else None

    comm_match = re.search(r"Communication:\s*(.+?)(?=\n|$)", triage_section)
    info["Communication"] = comm_match.group(1).strip() if comm_match else None

    # Chief Complaint and Condition
    chief_match = re.search(
        r"Chief complaint:\s*(.+?)(?=\nCondition|\nPast health:|\nTOCC:|\nReferral:|\nVital sign|\nTriage intervention|\nConsultation|$)",
        triage_section,
        re.DOTALL,
    )
    info["Chief complaint"] = chief_match.group(1).strip() if chief_match else None

    # Condition on arrival - capture multiple lines until Past health:
    condition_match = re.search(
        r"Condition on arrival:\s*(.*?)(?=\nPast health:|\nTOCC:|\nReferral:|\nVital sign|\nTriage intervention|\nConsultation|$)",
        triage_section,
        re.DOTALL,
    )
    info["Condition on arrival"] = (
        condition_match.group(1).strip() if condition_match else None
    )

    # Past Health and TOCC
    past_health_match = re.search(
        r"Past health:\s*(.+?)(?=\nTOCC:|\nReferral:|\nVital sign|\nTriage intervention|\nConsultation|$)",
        triage_section,
        re.DOTALL,
    )
    info["Past health"] = (
        past_health_match.group(1).strip() if past_health_match else None
    )

    tocc_match = re.search(r"TOCC:\s*(.+?)(?=\n|$)", triage_section)
    info["TOCC"] = tocc_match.group(1).strip() if tocc_match else None

    # Extract Referral
    referral_match = re.search(r"Referral:\s*(.+?)(?=\n|$)", triage_section)
    info["Referral"] = referral_match.group(1).strip() if referral_match else None

    # Use vital_section when available, fall back to original triage_section
    section_for_vitals = vital_section if vital_section else triage_section

    # Vital Signs
    gcs_match = re.search(
        r"GCS:\s*(E:\s*\d+,\s*V:\s*\d+,\s*M:\s*\d+,\s*Score:\s*\d+/\d+)",
        section_for_vitals,
    )
    info["GCS"] = gcs_match.group(1) if gcs_match else None

    # Enhanced BP extraction
    def extract_bp_readings(section_text):
        if not section_text:
            return None

        # First BP reading pattern - modified to better capture position
        first_bp_pattern = r"BP:\s*1st\s*(\d+)\s*/\s*(\d+)\s*mmHg,\s*([^,\n]+?)(?:\s*,\s*--\s*PR/AR|\s*,\s*--)"

        # Second BP reading pattern - modified to better capture position
        second_bp_pattern = (
            r"2nd\s*(\d+)\s*/\s*(\d+)\s*mmHg,\s*([^,\n]+?)(?:\s*,\s*--\s*2nd|\s*,\s*--)"
        )

        readings = []

        # Extract first reading
        first_match = re.search(first_bp_pattern, section_text)
        if first_match:
            systolic, diastolic, position = first_match.groups()
            position = position.strip()
            reading = f"1st: {systolic}/{diastolic} mmHg"
            if position and position != "--":
                reading += f" ({position})"
            readings.append(reading)

        # Extract second reading
        second_match = re.search(second_bp_pattern, section_text)
        if second_match:
            systolic, diastolic, position = second_match.groups()
            position = position.strip()
            reading = f"2nd: {systolic}/{diastolic} mmHg"
            if position and position != "--":
                reading += f" ({position})"
            readings.append(reading)

        return "; ".join(readings) if readings else None

    info["BP"] = extract_bp_readings(section_for_vitals)

    pr_match = re.search(r"PR/AR:\s*1st\s*(\d+)\s*/min", section_for_vitals)
    info["PR/AR"] = pr_match.group(1) if pr_match else None

    temp_match = re.search(r"Temp:\s*([\d.]+)°C", section_for_vitals)
    info["Temp"] = temp_match.group(1) if temp_match else None

    # Extract SpO2

    spo2_pattern = r"SpO\s*:?\s*(\d+)%\s*on\s*([^R]+?)(?=RR:|$)"
    spo2_match = re.search(spo2_pattern, section_for_vitals)
    if spo2_match:
        percentage, condition = spo2_match.groups()
        condition = condition.strip().rstrip(",").strip()

        # Fix for the "Non-rebreathing" without "mask" issue
        if "Non-rebreathing" in condition and "mask" not in condition.lower():
            condition = condition.replace("Non-rebreathing", "Non-rebreathing mask")

        info["SpO2"] = f"{percentage}% on {condition}"
    else:
        info["SpO2"] = None

    # Extract RR
    rr_pattern = r"RR:\s*((?:\d+(?:\s*/min)?)|(?:--))"
    rr_match = re.search(rr_pattern, section_for_vitals)
    if rr_match:
        rr_value = rr_match.group(1)
        if rr_value != "--":
            number_match = re.search(r"\d+", rr_value)
            if number_match:
                rr_value = number_match.group()
        info["RR"] = rr_value
    else:
        info["RR"] = None

    # Extract PFR
    pfr_pattern = r"PFR:\s*([^L\n]+?)(?=Limbs:|$)"
    pfr_match = re.search(pfr_pattern, section_for_vitals)
    info["PFR"] = pfr_match.group(1).strip() if pfr_match else None

    # Extract Limbs
    limbs_pattern = r"Limbs:\s*([^\n]+?)(?=\n|$)"
    limbs_match = re.search(limbs_pattern, section_for_vitals)
    info["Limbs"] = limbs_match.group(1).strip() if limbs_match else None

    def extract_pupil_info(section_text):
        if not section_text:
            return "--"

        # Look for pupil section
        pupil_pattern = r"Pupil:\s*Right\s*(\d+)\s*mm,\s*([^,\n]+?)(?:,\s*[^,\n]+)?(?:\n|,\s*VA).*?Left\s*(\d+)\s*mm,\s*([^,\n]+?)(?:,\s*[^,\n]+)?(?:\n|,\s*IOP)"

        pupil_match = re.search(pupil_pattern, section_text, re.DOTALL)

        if not pupil_match:
            return "--"

        # Extract measurements and reactivity
        right_size, right_react, left_size, left_react = pupil_match.groups()

        # Format the output
        right_info = f"Right: {right_size}mm {right_react.strip()}"
        left_info = f"Left: {left_size}mm {left_react.strip()}"

        return f"{right_info} | {left_info}"

    # Add pupil information
    info["Pupil"] = extract_pupil_info(section_for_vitals)

    def extract_pregnancy_info(section_text, sex):
        # Return N/A for males
        if sex == "M":
            return "N/A"

        if not section_text:
            return "unspecified"

        # Check if pregnancy section exists by looking for key fields
        pregnancy_section_indicators = [
            r"LMP:",
            r"Pregnant:",
            r"Menopause:",
            r"Gravida:",
            r"Para:",
        ]

        # Check if any of the pregnancy-related fields exist in the text
        section_exists = any(
            re.search(pattern, section_text) for pattern in pregnancy_section_indicators
        )

        if not section_exists:
            return "unspecified"

        # Extract all relevant fields with more precise patterns
        # The (?:(?!\w+:).)*? pattern ensures we don't capture text that belongs to another field
        pregnancy_fields = {
            "LMP": re.search(r"LMP:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$)", section_text),
            "PMP": re.search(r"PMP:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$)", section_text),
            "Pregnant": re.search(
                r"Pregnant:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$)", section_text
            ),
            "Maturity": re.search(
                r"Maturity:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$)", section_text
            ),
            "Menopause": re.search(
                r"Menopause:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$)", section_text
            ),
            "EDC": re.search(r"EDC:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$|\n)", section_text),
            "Gravida": re.search(r"Gravida:\s*(\d*)", section_text),
            "Para": re.search(r"Para:\s*(\d*)", section_text),
            "AN clinic": re.search(
                r"AN clinic:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$|\n)", section_text
            ),
        }

        # Clean up extracted values
        values = {}
        for k, v in pregnancy_fields.items():
            if v:
                value = v.group(1).strip()
                # Only include non-empty values
                if value and value != "--":
                    values[k] = value

        # Format the output: field1: value1 | field2: value2 | ...
        if values:
            formatted_items = [f"{k}: {v}" for k, v in values.items()]
            return " | ".join(formatted_items)

        return "unspecified"

    # Add pregnancy information
    info["Pregnancy Status"] = extract_pregnancy_info(section_for_vitals, info["Sex"])

    # Extract H'stix and H'cue from Triage intervention section
    def extract_triage_intervention_readings(text):
        # Find the triage intervention section
        triage_section = re.search(
            r"Triage intervention(.*?)Consultation", text, re.DOTALL
        )

        if not triage_section:
            return None, None

        triage_text = triage_section.group(1)

        # Extract H'stix
        hstix_match = re.search(r"H'stix:\s*([\d.]+)\s*mmol/L", triage_text)
        hstix = f"{hstix_match.group(1)} mmol/L" if hstix_match else None

        # Extract H'cue
        hcue_match = re.search(r"H'cue:\s*([\d.]+)\s*g/dL", triage_text)
        hcue = f"{hcue_match.group(1)} g/dL" if hcue_match else None

        return hstix, hcue

    # Add H'stix and H'cue to info dictionary
    info["H'stix"], info["H'cue"] = extract_triage_intervention_readings(text)

    # Discharge Information
    specialty_match = re.search(r"Attending specialty:\s*(.+?)(?=\n|$)", text)
    info["Attending specialty"] = (
        specialty_match.group(1).strip() if specialty_match else None
    )

    trauma_match = re.search(r"Trauma type:\s*(.+?)(?=\n|$)", text)
    info["Trauma type"] = trauma_match.group(1).strip() if trauma_match else None

    dest_match = re.search(r"Discharge destination:\s*(.+?)(?=\n|$)", text)
    info["Discharge destination"] = dest_match.group(1).strip() if dest_match else None

    return info


def process_directory_medical_notes(directory_path):
    """
    Process all text files in a specific directory and return a DataFrame

    Args:
        directory_path (Path): Path object pointing to directory with text files
    Returns:
        pd.DataFrame: Combined DataFrame with all medical records from the directory
    """
    all_records = {}

    # Process only txt files that end with _anonymized.txt
    for txt_file in directory_path.glob("*_anonymized.txt"):
        try:
            print(
                f" ================== Processing file: {txt_file} ================== "
            )
            # Read and process the file
            with open(txt_file, "r", encoding="utf-8") as file:
                text = file.read()

            # Extract information
            info_dict = extract_medical_info(text)

            # Add file path information
            info_dict["Source_File"] = str(txt_file.name)
            info_dict["Source_Directory"] = str(
                txt_file.parent.relative_to(directory_path.parent)
            )

            # Use filename as index
            all_records[txt_file.name] = info_dict

            print(
                f" ================== Successfully processed: {txt_file} ================== "
            )

        except Exception as e:
            print(
                f" ================== Error processing {txt_file}: {str(e)} ================== "
            )

    if all_records:
        # Convert to DataFrame
        df = pd.DataFrame.from_dict(all_records, orient="index")
        # Sort index (filenames) for consistency
        df.sort_index(inplace=True)
        return df
    return None

In [ ]:
def process_single_pdf_to_record(pdf_path):
    """
    Process a single PDF file and return a dictionary with all extracted information
    including AE number, medical data, and metadata
    """
    try:
        print(f"📄 Processing: {pdf_path.name}")
        
        # Step 1: Extract AE number
        ae_number = extract_ae_number_from_pdf(pdf_path)
        
        # Step 2: Extract and process text from all pages
        with pdfplumber.open(pdf_path) as pdf:
            all_cleaned_text = []
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    cleaned_text = anonymize_medical_note(text)
                    all_cleaned_text.append(cleaned_text)
            
            # Join all pages together
            final_text = "\n\n".join(all_cleaned_text)
            final_text = remove_repeating_headers(final_text)
        
        # Step 3: Extract medical information from processed text
        medical_info = extract_medical_info(final_text)
        
        # Step 4: Add metadata
        medical_info["AE_no"] = ae_number
        medical_info["Source_File"] = pdf_path.name  # PDF filename, not txt
        
        print(f"✅ Successfully processed: {pdf_path.name} (AE: {ae_number})")
        return medical_info
        
    except Exception as e:
        print(f"❌ Error processing {pdf_path}: {str(e)}")
        # Return minimal record with error info
        return {
            "AE_no": None,
            "Source_File": pdf_path.name,
            "Error": str(e)
        }

print("✅ Single PDF processing function defined")

In [ ]:
def batch_process_pdfs_to_csv(input_dir, output_dir):
    """
    Process all PDF files in input directory and its subdirectories,
    creating separate CSV files for each subdirectory in the output directory.
    Direct PDF → CSV conversion with AE number tracking.

    Args:
        input_dir (str): Path to directory containing PDF files
        output_dir (str): Path to directory where CSV files will be saved
    """
    # Convert to Path objects
    input_path = Path(input_dir)
    output_path = Path(output_dir)

    # Create output directory if it doesn't exist
    output_path.mkdir(parents=True, exist_ok=True)

    print(f"🚀 Starting batch processing...")
    print(f"📂 Input directory: {input_dir}")
    print(f"💾 Output directory: {output_dir}")
    print("="*70)

    # Dictionary to store records by directory
    directory_records = {}

    # Walk through all directories and files
    for root, dirs, files in os.walk(input_path):
        # Get the relative path from input directory
        rel_path = Path(root).relative_to(input_path)
        dir_key = str(rel_path) if rel_path != Path('.') else "root"
        
        # Process PDF files in current directory
        pdf_files = [f for f in files if f.lower().endswith(".pdf")]
        
        if pdf_files:
            print(f"\n📁 Processing directory: {dir_key} ({len(pdf_files)} PDF files)")
            print("-" * 50)
            
            directory_records[dir_key] = {}
            
            for pdf_file in pdf_files:
                pdf_path = Path(root) / pdf_file
                record = process_single_pdf_to_record(pdf_path)
                
                # Use filename as index (without .pdf extension)
                record_key = pdf_file[:-4]  # Remove .pdf extension
                directory_records[dir_key][record_key] = record

    # Convert to DataFrames and save CSV files
    print("\n" + "="*70)
    print("💾 Saving CSV files...")
    print("="*70)
    
    total_records = 0
    for dir_name, records in directory_records.items():
        if records:
            # Convert to DataFrame
            df = pd.DataFrame.from_dict(records, orient='index')
            
            # Sort by index (filenames) for consistency
            df.sort_index(inplace=True)
            
            # Move AE_no and Source_File to the front
            cols = df.columns.tolist()
            priority_cols = ['AE_no', 'Source_File']
            other_cols = [col for col in cols if col not in priority_cols]
            df = df[priority_cols + other_cols]
            
            # Create filename
            if dir_name == "root":
                filename = "root.csv"
            else:
                # Replace path separators with underscores for filename
                filename = f"{dir_name.replace(os.sep, '_')}.csv"
            
            # Save to CSV
            output_file = output_path / filename
            df.to_csv(output_file, index=True)
            
            total_records += len(df)
            
            print(f"✅ Saved {len(df)} records to: {output_file}")
            
            # Show sample of AE numbers found
            ae_numbers = df['AE_no'].dropna().tolist()[:5]
            if ae_numbers:
                print(f"   📝 Sample AE numbers: {ae_numbers}")
            
            # Show any files with missing AE numbers
            missing_ae = df[df['AE_no'].isna()]['Source_File'].tolist()
            if missing_ae:
                print(f"   ⚠️  Files missing AE numbers: {len(missing_ae)}")
                if len(missing_ae) <= 3:
                    print(f"      {missing_ae}")

    print(f"\n🎉 Processing complete!")
    print(f"📊 Total records processed: {total_records}")
    print(f"💾 CSV files saved to: {output_dir}")
    print(f"\n🔧 Key improvements implemented:")
    print(f"   ✅ AE numbers extracted and tracked")
    print(f"   ✅ Direct PDF → CSV conversion (no intermediate txt files)")
    print(f"   ✅ Source_File contains PDF filename")
    print(f"   ✅ Removed Source_Directory column")

print("✅ Batch processing function defined")

In [ ]:
batch_process_pdfs_to_csv(
    input_dir= root_dir + r"\pdf",
    output_dir=root_dir + r"\csv"
)

### 2.2 Manual CSV Review

Before converting to JSON, review the output CSV file(s) in `{root_dir}\csv\` to verify extraction quality.

Key fields to check:
- `Triage Category` — should be 1–5 for all records
- `Chief complaint` and `Condition on arrival` — should contain clinical text
- `BP`, `PR/AR`, `SpO2` — vital signs should be present for most records
- `Attending specialty` — used for case filtering during retrieval

Correct any missing or clearly incorrect values before proceeding to Section 2.3.

### 2.3 CSV to JSON Conversion

Converts the reviewed CSV files into per-case JSON files following the standard case schema. Each case is saved as an individual `.json` file.

After the batch conversion, run `post_process_json_files` to apply sex-specific field handling (removing or renaming the pregnancy status field based on patient sex).

In [ ]:
import pandas as pd
import json
import os
from pathlib import Path
import glob
import sys

In [ ]:
def format_value(value):
    """
    Properly format values:
    - Convert integers without decimal points
    - Return original values for non-numeric types
    - Handle NaN values
    """
    if pd.isna(value):
        return ""

    # Convert float to int if it's an integer value
    if isinstance(value, float) and value.is_integer():
        return str(int(value))

    # Return string representation for any other value
    return str(value)


def convert_csv_to_json_with_tracking(csv_file_path, output_dir):
    """
    Convert a CSV file with medical triage data to JSON format.

    Args:
        csv_file_path (str): Path to the CSV file
        output_dir (str): Directory to save JSON files

    Returns:
        int: Number of records processed
    """
    # Read the CSV file with more control over data types
    df = pd.read_csv(csv_file_path, index_col=0)

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Add new column for JSON paths
    df['JSON_Path'] = ''

    records_processed = 0

    # Process each row in the dataframe
    for index, row in df.iterrows():
        # Create the JSON structure
        record = {
            "Demographics": {
                "Sex": format_value(row.get("Sex", "")),
                "Age": format_value(row.get("Age", "")),
                "Ambulatory status": format_value(row.get("Ambulatory status", "")),
                "Risk of fall": format_value(row.get("Risk of fall", "")),
                "Informant": format_value(row.get("Informant", "")),
                "Communication": format_value(row.get("Communication", "")),
                "Allergies": format_value(row.get("Allergy", "")),
                "ADR": format_value(row.get("ADR", "")),
                "Alerts": format_value(row.get("Alert", "")),
                "Past health": format_value(row.get("Past health", "")),
                "TOCC": format_value(row.get("TOCC", "")),
                "Referral (if any)": format_value(row.get("Referral", "")),
            },
            "Clinical Presentation": {
                "Chief complaint": format_value(row.get("Chief complaint", "")),
                "Condition on Arrival": format_value(
                    row.get("Condition on arrival", "")
                ),
            },
            "Vitals and Observations": {
                "Vital Signs": {
                    "GCS": format_value(row.get("GCS", "")),
                    "BP": format_value(row.get("BP", "")),
                    "PR/AR": format_value(row.get("PR/AR", "")),
                    "Temp": format_value(row.get("Temp", "")),
                    "SpO2": format_value(row.get("SpO2", "")),
                    "RR": format_value(row.get("RR", "")),
                    "PFR": format_value(row.get("PFR", "")),
                    "Limbs": format_value(row.get("Limbs", "")),
                    "Pupil": format_value(row.get("Pupil", "")),
                },
                "Pregnancy Status": format_value(row.get("Pregnancy Status", "")),
                "Triage Intervention": {
                    "H'stix": format_value(row.get("H'stix", "")),
                    "H'cue": format_value(row.get("H'cue", "")),
                },
            },
            "Case Disposition": {
                "Triage Category": format_value(row.get("Triage Category", "")),
                "Diagnosis": format_value(row.get("Diagnosis", "")),
                "Attending Specialty": format_value(row.get("Attending specialty", "")),
                "Trauma Type": format_value(row.get("Trauma type", "")),
                "Discharge Destination": format_value(
                    row.get("Discharge destination", "")
                ),
            },
        }

        # FIXED: Handle PDF filenames properly
        source_file = row.get("Source_File", f"case_{index}.pdf")

        # Remove .pdf extension and add .json
        if source_file.endswith('.pdf'):
            output_filename = source_file.replace('.pdf', '.json')
        else:
            # Fallback for any unexpected filename format
            output_filename = f"{source_file}.json"

        # Create full output path
        output_path = os.path.join(output_dir, output_filename)

        # Save the JSON
        with open(output_path, "w", encoding="utf-8") as json_file:
            json.dump(record, json_file, indent=4, ensure_ascii=False)

        # Add JSON path to DataFrame
        df.at[index, 'JSON_Path'] = output_path
        
        records_processed += 1

    return df, records_processed


def batch_process_directory_with_tracking(input_dir, output_dir):
    """
    Process all CSV files and update them with JSON_Path column
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    
    # Ensure output directory exists
    os.makedirs(output_path, exist_ok=True)
    
    # Find all CSV files
    csv_files = list(input_path.glob("*.csv"))
    
    if not csv_files:
        print(f"No CSV files found in {input_dir}")
        return
    
    total_files = len(csv_files)
    total_records = 0
    
    print(f"Found {total_files} CSV files to process.")
    
    # Process each CSV file
    for i, csv_file in enumerate(csv_files, 1):
        try:
            # Create subfolder named after CSV file
            subfolder_name = csv_file.stem
            subfolder_path = output_path / subfolder_name
            
            print(f"[{i}/{total_files}] Processing {csv_file.name} => {subfolder_name}/")
            
            # Convert CSV to JSON and get updated DataFrame
            updated_df, records = convert_csv_to_json_with_tracking(
                str(csv_file), str(subfolder_path)
            )
            
            # Save updated CSV with JSON_Path column
            updated_df.to_csv(csv_file, index=True)
            
            total_records += records
            print(f"  ✓ Processed {records} records from {csv_file.name}")
            print(f"  ✓ Updated {csv_file.name} with JSON_Path column")
            
        except Exception as e:
            print(f"  ✗ Error processing {csv_file.name}: {str(e)}")
    
    print(f"\nProcessing complete!")
    print(f"Total CSV files processed: {total_files}")
    print(f"Total records converted: {total_records}")
    print(f"All CSV files updated with JSON_Path column")
    print(f"JSON files saved to: {output_dir}")

def post_process_json_files(output_base):
    """
    Post-process all formatted JSON files to:
    1. Remove 'Pregnancy Status' field if Sex is 'M'
    2. Rename 'Pregnancy Status' to 'Menstruation / Pregnancy Status' for non-male patients

    Args:
        output_base (str or Path): Path to the output directory containing formatted JSON files
    """
    # Convert output_base to a Path object if it's a string
    output_base = Path(output_base) if isinstance(output_base, str) else output_base

    print("\n" + "=" * 50)
    print("STARTING POST-PROCESSING")
    print("=" * 50)

    total_files = 0
    modified_files = 0

    # Process each category directory in the output folder
    for category_dir in output_base.glob("*"):
        if category_dir.is_dir():
            print(f"\nPost-processing directory: {category_dir.name}")

            # Process each formatted JSON file in the category directory
            for json_file in category_dir.glob("*.json"):
                total_files += 1

                try:
                    print(f"Post-processing: {json_file.name}")

                    # Load JSON data
                    with open(json_file, "r", encoding="utf-8") as f:
                        json_data = json.load(f)

                    modified = False

                    # Check if Demographics and Sex fields exist
                    if (
                        "Demographics" in json_data
                        and "Sex" in json_data["Demographics"]
                    ):
                        sex = json_data["Demographics"]["Sex"]

                        # Check if Vitals and Observations section exists
                        if "Vitals and Observations" in json_data:
                            vitals_section = json_data["Vitals and Observations"]

                            # Case 1: Remove Pregnancy Status for male patients
                            # Check for 'M' or any case variant of 'male'
                            if sex == "M" or sex.lower() == "male":
                                if "Pregnancy Status" in vitals_section:
                                    del vitals_section["Pregnancy Status"]
                                    modified = True
                                    print(
                                        f"  - Removed 'Pregnancy Status' field (patient is male)"
                                    )
                                if "Menstruation / Pregnancy Status" in vitals_section:
                                    del vitals_section[
                                        "Menstruation / Pregnancy Status"
                                    ]
                                    modified = True
                                    print(
                                        f"  - Removed 'Menstruation / Pregnancy Status' field (patient is male)"
                                    )
                            # Case 2: Rename Pregnancy Status for non-male patients
                            # Check if not 'M' and not any case variant of 'male'
                            elif (
                                sex != "M" and sex.lower() != "male"
                            ) and "Pregnancy Status" in vitals_section:
                                # Create new key with the old value
                                vitals_section["Menstruation / Pregnancy Status"] = (
                                    vitals_section["Pregnancy Status"]
                                )
                                # Delete the old key
                                del vitals_section["Pregnancy Status"]
                                modified = True
                                print(
                                    f"  - Renamed 'Pregnancy Status' to 'Menstruation / Pregnancy Status'"
                                )

                    # Save modified JSON data if changes were made
                    if modified:
                        with open(json_file, "w", encoding="utf-8") as f:
                            json.dump(json_data, f, indent=4)
                        modified_files += 1
                        print(f"  - Saved modifications to {json_file.name}")
                    else:
                        print(f"  - No modifications needed for {json_file.name}")

                except Exception as e:
                    print(f"Error post-processing file {json_file.name}: {str(e)}")
                    continue

    print(f"\nPost-processing complete. Files modified: {modified_files}/{total_files}")

In [ ]:
batch_process_directory_with_tracking(
    input_dir=root_dir + r"\csv", 
    output_dir=root_dir + r"\json"
)

In [ ]:
post_process_json_files(
    output_base=root_dir + r"\json"
)